# Lesson 7: The averaging trick (the first step toward attention)

**Why:** the bigram model failed because each character saw only **itself**. We want each position to **gather information from all earlier positions**, but **never from later ones**. When writing text, the future doesn't exist yet, so peeking at it during training would be cheating.

```
position 0 may use:  x0
position 1 may use:  x0  x1
position 2 may use:  x0  x1  x2
position 3 may use:  x0  x1  x2  x3        never anything to its right
```

The simplest way to gather: **average** everything up to and including yourself. It's crude, because every earlier character counts equally, but it's the skeleton of attention. Today we compute it **three ways**, each faster and more flexible than the last.

## 1. Version 1: plain loops

A tiny example: 1 chunk, 4 positions, 2 numbers per position. The numbers are simple so you can check the averages in your head.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)

x = torch.tensor([[[1.0, 2.0],
                   [3.0, 4.0],
                   [5.0, 6.0],
                   [7.0, 8.0]]])          # (B=1, T=4, C=2)
B, T, C = x.shape

out1 = torch.zeros_like(x)
for b in range(B):
    for t in range(T):
        out1[b, t] = x[b, : t + 1].mean(dim=0)   # average of positions 0..t

print(out1)

## 2. Version 2: one matrix multiply

Loops are slow. The trick: a **lower-triangular** matrix, which has ones on and below the diagonal and zeros above. `torch.tril` builds it.

**Predict:** what does `torch.tril(torch.ones(4, 4))` look like? Then, after we divide each row by its sum, what is row 2?

In [ ]:
tril = torch.tril(torch.ones(T, T))
print("tril:")
print(tril)

wei = tril / tril.sum(dim=1, keepdim=True)   # each row now adds up to 1
print("\nweights (each row = how much of each position to mix in):")
print(wei)

out2 = wei @ x        # (T, T) @ (B, T, C) -> (B, T, C)
print("\nresult:")
print(out2)

**Why it works:** row *t* of `wei` holds the averaging recipe for position *t*: equal shares of positions 0..t, and **zero** for everything after. Multiplying `wei @ x` applies every row's recipe at once (Lesson 1: the T×T matrix is broadcast across the batch).

## 3. Version 3: softmax with a mask (the one attention uses)

Same result, built a different way:
1. Start with **scores** for every pair of positions (all zeros for now, meaning "everyone equally interesting").
2. Set every **future** position's score to **−∞**.
3. **Softmax** each row (Lesson 3). Since $e^{-\infty} = 0$, the future gets exactly zero weight, and the rest share equally.

Why bother? In Lesson 8, the scores **won't be zeros**. They'll be learned answers to *"how relevant is that earlier character to me?"* This version handles that for free.

In [ ]:
scores = torch.zeros(T, T)
scores = scores.masked_fill(tril == 0, float("-inf"))   # block the future
print("scores after masking:")
print(scores)

wei3 = F.softmax(scores, dim=-1)
out3 = wei3 @ x
print("\nweights after softmax:")
print(wei3)

## 4. Are all three the same? And a preview of attention

Check on bigger random data, then see what happens when the scores are **not** all equal. The heatmaps show the weights: darker means more attention. The upper-right triangle is the blocked future.

In [ ]:
xb = torch.randn(4, 8, 32)                              # (B, T, C), a realistic shape
Tb = xb.shape[1]
tril8 = torch.tril(torch.ones(Tb, Tb))

loop = torch.stack([torch.stack([xb[b, : t + 1].mean(0) for t in range(Tb)]) for b in range(4)])
matmul = (tril8 / tril8.sum(1, keepdim=True)) @ xb
softmax = F.softmax(torch.zeros(Tb, Tb).masked_fill(tril8 == 0, float("-inf")), dim=-1) @ xb
print("loop == matmul: ", torch.allclose(loop, matmul, atol=1e-6))
print("loop == softmax:", torch.allclose(loop, softmax, atol=1e-6))

uneven = F.softmax(torch.randn(Tb, Tb).masked_fill(tril8 == 0, float("-inf")), dim=-1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 4))
a1.imshow(F.softmax(torch.zeros(Tb, Tb).masked_fill(tril8 == 0, float("-inf")), -1), cmap="Blues")
a1.set_title("Today: plain average"); a1.set_xlabel("looks at position"); a1.set_ylabel("position")
a2.imshow(uneven, cmap="Blues")
a2.set_title("Lesson 8: uneven (learned) attention"); a2.set_xlabel("looks at position")
plt.show()

## 5. Final check

In [ ]:
assert torch.allclose(out1, out2) and torch.allclose(out1, out3)
assert torch.allclose(loop, softmax, atol=1e-6)
assert torch.all(wei3.triu(1) == 0), "no weight may go to the future"
print("All three versions agree, and the future is blocked. Lesson 7 complete.")

## Wrap-up question

Why do we set the future scores to **−∞** before softmax, instead of setting them to **0**?